# AutoEncoder : compression, espace latent et generation

In [ ]:
from src.dataset import load_mnist_dataset, load_shapes_npz
from src.autoencoder import AutoEncoder
from src.helper import extract_full_dataset, get_device
from src.metrics import compression_report, Latent

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

np.random.seed(0)
torch.manual_seed(0)

print("device:", get_device())


EPOCHS = 40
EPOCHS_SWEEP = 40
BATCH_SIZE = 128

In [ ]:
# Les helpers de visualisation, communs aux notebooks, vivent maintenant dans src/viz.py.
# Ne restent ici que les deux fonctions specifiques a l'AutoEncoder.
# finish_figure garde son ancien nom local pour ne pas toucher les cellules existantes.
from src.viz import (
    flattened_vector_to_image,
    finish_figure as _finish_figure,
    format_fixed_note,
    show_original_vs_reconstruction_grid,
    show_image_grid,
    show_labeled_image_rows,
    plot_latent_scatter,
    print_compression_report,
    sample_gaussian_latent,
    generate_from_latent_using_gaussian,
    interpolate_latent,
    subsample_dataset,
)


def describe_autoencoder(model, n_train=None, omit=(), extra=None):
    linears = [layer for layer in model.encoder.net if isinstance(layer, nn.Linear)]
    sizes = [linears[0].in_features] + [layer.out_features for layer in linears]
    # L'activation cachee est cherchee UNIQUEMENT avant le dernier Linear: sinon une
    # latent_activation, ajoutee en fin d'encodeur, serait prise pour une activation cachee.
    last_linear = max(i for i, layer in enumerate(model.encoder.net) if isinstance(layer, nn.Linear))
    activation = next((type(layer).__name__ for layer in list(model.encoder.net)[:last_linear]
                       if not isinstance(layer, nn.Linear)), "aucune")
    fields = {
        "couches": "-".join(str(size) for size in sizes),
        "act": activation,
        "latent_act": model.latent_activation.__name__ if model.latent_activation else "aucune",
        "out_act": model.output_activation.__name__ if model.output_activation else "aucune",
        "loss": model.fonction_loss.__name__,
        "epochs": str(len(model.loss_history)),
    }
    if n_train is not None:
        fields["n_train"] = f"{n_train:,}"
    return format_fixed_note(fields, omit, extra)


def run_autoencoder_hyperparam_experiment(
        X_train, X_eval, input_dim, latent_dim, activation_function, epochs, loss_function=nn.MSELoss,
        latent_activation_function=None, output_activation_function=nn.Sigmoid,
        encoder_layer_num=3, decoder_layer_num=3, learning_rate : float = 1e-3
        ):
    model = AutoEncoder(
        input_dim=input_dim, output_dim=input_dim, latent_dim=latent_dim,
        encoder_layer_num=encoder_layer_num, decoder_layer_num=decoder_layer_num,
        encoder_activation_function=activation_function, loss_function=loss_function,
        latent_activation_function=latent_activation_function, output_activation_function=output_activation_function
    )
    model.fit(X_train, epochs=epochs, batch_size=BATCH_SIZE, learning_rate=learning_rate)
    latent = model.encode(X_eval)
    reconstruction = model.decode(latent)
    report = compression_report(model.get_codebook(), latent, X_eval, reconstruction)
    return {"model": model, "latent": latent, "reconstruction": reconstruction, "report": report}

## Partie A - MNIST DIGITS

In [ ]:
mnist_train_images, mnist_train_labels = extract_full_dataset(load_mnist_dataset(train=True, shuffle=False))
mnist_eval_images, mnist_eval_labels = extract_full_dataset(load_mnist_dataset(train=False, shuffle=False))

MNIST_SHAPE = (1, 28, 28)
X_mnist_train, y_mnist_train = subsample_dataset(
    mnist_train_images.reshape(len(mnist_train_images), -1).numpy(), mnist_train_labels.numpy(), 15000
    )
X_mnist_eval, y_mnist_eval = subsample_dataset(
    mnist_eval_images.reshape(len(mnist_eval_images), -1).numpy(), mnist_eval_labels.numpy(), 3000
    )
print("train:", X_mnist_train.shape, "| eval:", X_mnist_eval.shape)

### Train - ReLu et MSE comme baseline

In [ ]:
mnist_model = AutoEncoder(
    input_dim=784, output_dim=784, latent_dim=2,
    encoder_layer_num=3, decoder_layer_num=3, encoder_activation_function=nn.ReLU, loss_function=nn.MSELoss
)
mnist_model.fit(X_mnist_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

MNIST_BASELINE_CONFIG = describe_autoencoder(mnist_model, n_train=len(X_mnist_train))

fig = plt.figure(figsize=(5, 3))
plt.plot(mnist_model.loss_history, marker="o")
plt.xlabel("epoch"); plt.ylabel("MSE d'entrainement")
# epochs varie le long de l'axe x: on le retire de la note
_finish_figure(fig, "Courbe d'apprentissage MNIST",
               describe_autoencoder(mnist_model, n_train=len(X_mnist_train), omit=("epochs",)))

### Compression et decompression

In [ ]:
mnist_latent = mnist_model.encode(X_mnist_eval)
mnist_reconstructed = mnist_model.decode(mnist_latent)
print("code latent par image:", mnist_latent.array.shape[1], "valeurs | nature:", mnist_latent.nature)
show_original_vs_reconstruction_grid(X_mnist_eval, mnist_reconstructed, MNIST_SHAPE, n=8,
                     title="MNIST: original (haut) vs reconstruit (bas)",
                     config=MNIST_BASELINE_CONFIG)

### Qualite de reconstruction et taille du message

In [ ]:
mnist_report = compression_report(mnist_model.get_codebook(), mnist_latent, X_mnist_eval, mnist_reconstructed)
print_compression_report(mnist_report)

per_image_bytes = mnist_latent.n_bytes / len(X_mnist_eval)
print(f"\nMessage transmis par image: {per_image_bytes:.0f} octets ({mnist_latent.array.shape[1]} float32),")
print(f"contre {X_mnist_eval[0].nbytes} octets pour l'image originale en float32.")
print(f"Codebook (poids du decodeur): {mnist_report['codebook_bytes']:,} octets, partage une seule fois.")

### Visualisation de l'espace latent

In [ ]:
plot_latent_scatter(mnist_latent.array, y_mnist_eval,
                    title="Espace latent 2D des chiffres MNIST",
                    config=MNIST_BASELINE_CONFIG)

### Generation de nouvelles images

Le decodeur seul est un generateur: on lui fournit des codes latents inedits.

- Echantillonnage gaussien: on ajuste une gaussienne sur les codes du jeu d'evaluation et on en tire de nouveaux.
- Interpolation: on relie deux images reelles par une droite dans l'espace latent.
- Balayage de la variete 2D: on decode une grille reguliere du plan latent du modele 2D.

Note: un AutoEncoder classique ne contraint pas son espace latent a une loi connue; rester pres de la distribution des donnees (gaussienne ajustee, interpolation) donne les images les plus nettes.

In [ ]:
# Echantillonnage gaussien
new_codes = sample_gaussian_latent(mnist_latent.array, n_samples=32)
generated = mnist_model.decode(Latent(array=new_codes, nature="continuous"))
show_image_grid(generated, MNIST_SHAPE, nrow=4, ncol=8,
                title="Chiffres generes (echantillonnage gaussien)",
                config=MNIST_BASELINE_CONFIG)

# Interpolation entre deux images
path = interpolate_latent(mnist_latent.array[0], mnist_latent.array[1], steps=10)
morph = mnist_model.decode(Latent(array=path, nature="continuous"))
show_image_grid(morph, MNIST_SHAPE, nrow=1, ncol=10,
                title="Interpolation dans l'espace latent",
                config=MNIST_BASELINE_CONFIG)

# Balayage du latent
lo, hi = mnist_latent.array.min(axis=0), mnist_latent.array.max(axis=0)
grid_axis = 12
xs = np.linspace(lo[0], hi[0], grid_axis)
ys = np.linspace(hi[1], lo[1], grid_axis)
grid = np.array([[x, y] for y in ys for x in xs], dtype=np.float32)
manifold = mnist_model.decode(Latent(array=grid, nature="continuous"))
show_image_grid(manifold, MNIST_SHAPE, nrow=grid_axis, ncol=grid_axis,
                title="Variete generee par le plan latent 2D",
                config=MNIST_BASELINE_CONFIG)


# Balayage du latent avec encore plus de détails
lo, hi = mnist_latent.array.min(axis=0), mnist_latent.array.max(axis=0)
grid_axis = 24
xs = np.linspace(lo[0], hi[0], grid_axis)
ys = np.linspace(hi[1], lo[1], grid_axis)
grid = np.array([[x, y] for y in ys for x in xs], dtype=np.float32)
manifold = mnist_model.decode(Latent(array=grid, nature="continuous"))
show_image_grid(manifold, MNIST_SHAPE, nrow=grid_axis, ncol=grid_axis,
                title="Variete generee par le plan latent 2D",
                config=MNIST_BASELINE_CONFIG)

### Experimentation sur les hyper-parametres - courbes

In [ ]:
latent_dims = [2, 8, 16, 32, 64]
activations = {"ReLU": nn.ReLU, "Tanh": nn.Tanh}
X_tr, y_tr = subsample_dataset(X_mnist_train, y_mnist_train, 6000, seed=1)

mnist_runs = {}
mnist_results = []
for act_name, act in activations.items():
    for latent_dim in latent_dims:
        if act_name == "Tanh":
            run = run_autoencoder_hyperparam_experiment(
                X_tr, X_mnist_eval, 784, latent_dim, act, EPOCHS_SWEEP*10, learning_rate=1e-4)
        else:
            run = run_autoencoder_hyperparam_experiment(X_tr, X_mnist_eval, 784, latent_dim, act, EPOCHS_SWEEP)
        mnist_runs[(act_name, latent_dim)] = run
        report = run["report"]
        report.update(latent_dim=latent_dim, activation=act_name)
        mnist_results.append(report)
        print(f"{act_name:>5} | latent={latent_dim:>2} | MSE={report['reconstruction_mse']:.4f} "
              f"| total={report['total_compressed_bytes']:>9,} o | ratio={report['compression_ratio']:.2f}")

# Explore ici: la dimension latente (axe x) et l'activation (legende) -> retirees de la note.
# Les couches suivent la dimension latente, on donne donc la regle plutot qu'une taille.
MNIST_SWEEP_CONFIG = describe_autoencoder(
    mnist_runs[("ReLU", latent_dims[0])]["model"], n_train=len(X_tr),
    omit=("couches", "act"), extra="couches=geomspace(784->latent, 3+3)",
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for act_name in activations:
    pts = [r for r in mnist_results if r["activation"] == act_name]
    axes[0].plot([r["latent_dim"] for r in pts], [r["reconstruction_mse"] for r in pts], marker="o", label=act_name)
    axes[1].plot([r["total_compressed_bytes"] for r in pts], [r["reconstruction_mse"] for r in pts], marker="o", label=act_name)
axes[0].set_xlabel("dimension latente"); axes[0].set_ylabel("MSE"); axes[0].set_title("Qualite vs dimension latente"); axes[0].legend()
axes[1].set_xlabel("taille compressee totale (octets)"); axes[1].set_ylabel("MSE"); axes[1].set_title("Compromis taille / qualite"); axes[1].legend()
_finish_figure(fig, config=MNIST_SWEEP_CONFIG)

### Experimentation sur les hyper-parametres - images

In [ ]:
VIZ_ACTIVATION = "ReLU"

# Explore ici: la dimension latente (etiquettes de lignes). Tout le reste est fixe et va dans la note.
latent_sweep_config = describe_autoencoder(
    mnist_runs[(VIZ_ACTIVATION, latent_dims[0])]["model"], n_train=len(X_tr),
    omit=("couches",), extra="couches=geomspace(784->latent, 3+3)",
)

show_labeled_image_rows(
    [X_mnist_eval] + [mnist_runs[(VIZ_ACTIVATION, d)]["reconstruction"] for d in latent_dims],
    MNIST_SHAPE,
    ["original"] + [f"latent={d}\n{mnist_runs[(VIZ_ACTIVATION, d)]['report']['reconstruction_mse']:.4f}" for d in latent_dims],
    n=8, title="MNIST: decompression selon la dimension latente (etiquette = MSE)",
    config=latent_sweep_config,
)

show_labeled_image_rows(
    [generate_from_latent_using_gaussian(mnist_runs[(VIZ_ACTIVATION, d)]["model"], mnist_runs[(VIZ_ACTIVATION, d)]["latent"], 8, seed=1)
     for d in latent_dims],
    MNIST_SHAPE,
    [f"latent={d}" for d in latent_dims],
    n=8, title="MNIST: generation par echantillonnage gaussien selon la dimension latente",
    config=latent_sweep_config,
)

#### Espace latent selon la fonction d'activation


In [ ]:
latent_viz_activations = {"ReLU": nn.ReLU, "Tanh": nn.Tanh, "Sigmoid": nn.Sigmoid, "LeakyReLU": nn.LeakyReLU}

for act_name, act in latent_viz_activations.items():
    if (act_name, 2) not in mnist_runs:
        mnist_runs[(act_name, 2)] = run_autoencoder_hyperparam_experiment(
            X_tr, X_mnist_eval, 784, 2, act, EPOCHS_SWEEP
        )
        
activation_viz_config = describe_autoencoder(
    mnist_runs[("ReLU", 2)]["model"], n_train=len(X_tr), omit=("act",)
)

fig, axes = plt.subplots(1, len(latent_viz_activations),
                         figsize=(3.6 * len(latent_viz_activations), 3.9), layout="constrained")
for ax, act_name in zip(axes, latent_viz_activations):
    run = mnist_runs[(act_name, 2)]
    codes = run["latent"].array
    scatter = ax.scatter(codes[:, 0], codes[:, 1], c=y_mnist_eval, cmap="tab10", s=4, alpha=0.6)
    ax.set_title(f"{act_name} (MSE={run['report']['reconstruction_mse']:.4f})", fontsize=10)
    ax.set_xlabel("z1"); ax.set_ylabel("z2")
fig.colorbar(scatter, ax=list(axes), label="chiffre")
_finish_figure(fig, "MNIST: espace latent 2D selon la fonction d'activation",
               activation_viz_config, layout=False)

show_labeled_image_rows(
    [generate_from_latent_using_gaussian(mnist_runs[(act_name, 2)]["model"], mnist_runs[(act_name, 2)]["latent"], 8, seed=1)
     for act_name in latent_viz_activations],
    MNIST_SHAPE,
    list(latent_viz_activations),
    n=8, title="MNIST: generation depuis un latent 2D selon la fonction d'activation",
    config=activation_viz_config,
)

### Variation de la perte et de l'activation de sortie

La perte et l'activation de sortie du decodeur ne sont pas deux choix independants: elles
doivent etre compatibles. On croise donc les deux axes plutot que de faire varier la perte
seule.

`nn.BCELoss` tolere exactement 0 et 1 (son `log` est borne a -100) mais **rejette toute valeur
superieure a 1**. Une sortie non bornee (`ReLU`, ou aucune activation) construit donc sans
erreur, les premiers batchs passent, puis l'assertion echoue en pleine boucle d'entrainement.
Sur GPU c'est pire: l'assertion est levee cote device et **corrompt definitivement le contexte
CUDA**, si bien que toutes les cellules suivantes echouent jusqu'au redemarrage du kernel.
`AutoEncoder` refuse donc ces couples des la construction, avant toute operation CUDA.

Deux cases de la matrice sont donc structurellement impossibles, et c'est la moitie de la
lecon: `ReLU + BCE` et `sortie lineaire + BCE`. L'alternative correcte pour une sortie non
bornee est `nn.BCEWithLogitsLoss`, qui applique la sigmoide en interne de facon
numeriquement stable.

In [ ]:
loss_functions = {"MSE": nn.MSELoss, "L1": nn.L1Loss, "BCE": nn.BCELoss}
output_activations = {"Sigmoid": nn.Sigmoid, "ReLU": nn.ReLU, "aucune": None}
LOSS_EXP_LATENT = 32

# Couples impossibles: BCE exige des sorties dans [0,1]. On les saute explicitement plutot
# que d'attraper une exception: sur GPU l'assertion de BCELoss est levee cote device et
# corromprait le contexte CUDA pour toutes les cellules suivantes.
COUPLES_INVALIDES = {("ReLU", "BCE"), ("aucune", "BCE")}

mnist_out_loss = {}
loss_config = None
for out_name, out_act in output_activations.items():
    for loss_name, loss_cls in loss_functions.items():
        if (out_name, loss_name) in COUPLES_INVALIDES:
            print(f"out_act={out_name:>8} | perte={loss_name:<3} | INVALIDE (sortie non bornee + BCE)")
            continue
        run = run_autoencoder_hyperparam_experiment(
            X_tr, X_mnist_eval, 784, LOSS_EXP_LATENT, nn.ReLU, EPOCHS_SWEEP,
            loss_function=loss_cls, output_activation_function=out_act,
        )
        mse = run["report"]["reconstruction_mse"]
        mnist_out_loss[(out_name, loss_name)] = run
        # Part des pixels reconstruits exactement nuls: revelateur pour ReLU en sortie.
        zeros = 100 * np.mean(run["reconstruction"] == 0)
        print(f"out_act={out_name:>8} | perte={loss_name:<3} | MSE={mse:.4f} | pixels exactement nuls: {zeros:5.1f}%")

# Explore ici: l'activation de sortie (lignes) et la perte (colonnes) -> retirees de la note.
loss_config = describe_autoencoder(
    mnist_out_loss[("Sigmoid", "MSE")]["model"], n_train=len(X_tr), omit=("loss", "out_act")
)

# Carte annotee: un diagramme en barres ne saurait pas montrer l'ABSENCE des couples invalides.
matrice = np.full((len(output_activations), len(loss_functions)), np.nan)
for i, out_name in enumerate(output_activations):
    for j, loss_name in enumerate(loss_functions):
        if (out_name, loss_name) in mnist_out_loss:
            matrice[i, j] = mnist_out_loss[(out_name, loss_name)]["report"]["reconstruction_mse"]

fig, ax = plt.subplots(figsize=(5.5, 3.6))
image = ax.imshow(np.ma.masked_invalid(matrice), cmap="viridis_r")
image.cmap.set_bad("0.85")
ax.set_xticks(range(len(loss_functions)), list(loss_functions))
ax.set_yticks(range(len(output_activations)), list(output_activations))
ax.set_xlabel("fonction de perte"); ax.set_ylabel("activation de sortie")
for i, out_name in enumerate(output_activations):
    for j, loss_name in enumerate(loss_functions):
        valeur = matrice[i, j]
        texte = "invalide" if np.isnan(valeur) else f"{valeur:.4f}"
        couleur = "0.35" if np.isnan(valeur) else "white"
        ax.text(j, i, texte, ha="center", va="center", fontsize=9, color=couleur)
fig.colorbar(image, ax=ax, label="MSE d'evaluation")
_finish_figure(fig, "MNIST: MSE d'evaluation selon sortie x perte", loss_config)

# La ligne ReLU vs Sigmoid a perte identique: c'est la que les pixels morts se voient.
show_labeled_image_rows(
    [X_mnist_eval] + [mnist_out_loss[(o, "MSE")]["reconstruction"] for o in output_activations],
    MNIST_SHAPE,
    ["original"] + [f"{o}\n{mnist_out_loss[(o, 'MSE')]['report']['reconstruction_mse']:.4f}" for o in output_activations],
    n=8, title="MNIST: decompression selon l'activation de sortie, perte MSE (etiquette = MSE)",
    config=describe_autoencoder(mnist_out_loss[("Sigmoid", "MSE")]["model"], n_train=len(X_tr), omit=("out_act",)),
)

#### Neurones morts avec ReLU en sortie

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

toujours_noir = (X_mnist_eval == 0).all(axis=0)
print(f"Pixels toujours noirs dans les donnees d'evaluation: {toujours_noir.sum()}/784 "
      f"({100 * toujours_noir.mean():.1f}%) -> y sortir 0 est correct.\n")

for out_name in output_activations:
    run = mnist_out_loss[(out_name, "MSE")]
    mort_partout = (run["reconstruction"] == 0).all(axis=0)
    vraiment_morts = mort_partout & ~toujours_noir
    erreur = ((X_mnist_eval - run["reconstruction"]) ** 2).sum(axis=0)
    part_erreur = 100 * erreur[vraiment_morts].sum() / erreur.sum() if vraiment_morts.any() else 0.0
    avec_encre = 100 * np.mean((X_mnist_eval[:, vraiment_morts] > 0).any(axis=1)) if vraiment_morts.any() else 0.0
    print(f"{out_name:>8} | MSE={run['report']['reconstruction_mse']:.4f} "
          f"| morts bruts={mort_partout.sum():>3} | VRAIMENT morts={vraiment_morts.sum():>3} "
          f"| part de l'erreur totale={part_erreur:5.1f}% | images touchees={avec_encre:5.1f}%")

# Carte detaillee de la sortie ReLU, la seule qui presente la pathologie.
relu_run = mnist_out_loss[("ReLU", "MSE")]
mort_partout = (relu_run["reconstruction"] == 0).all(axis=0)
vraiment_morts = mort_partout & ~toujours_noir
erreur = ((X_mnist_eval - relu_run["reconstruction"]) ** 2).sum(axis=0)

# 0 = pixel vivant, 1 = toujours noir dans les donnees (0 est correct), 2 = vraiment mort
categories = np.zeros(784)
categories[toujours_noir] = 1
categories[vraiment_morts] = 2

fig, axes = plt.subplots(1, 3, figsize=(11, 3.9), layout="constrained")
axes[0].imshow(toujours_noir.reshape(28, 28), cmap="gray_r")
axes[0].set_title("Donnees: pixels toujours noirs\n(0 est la bonne reponse)", fontsize=9)

cmap = ListedColormap(["0.15", "0.55", "#d62728"])
axes[1].imshow(categories.reshape(28, 28), cmap=cmap, vmin=0, vmax=2)
axes[1].set_title(f"Sortie ReLU: {vraiment_morts.sum()} pixels vraiment morts", fontsize=9)
axes[1].legend(handles=[
    Patch(facecolor="0.15", label="vivant"),
    Patch(facecolor="0.55", label="toujours noir (ok)"),
    Patch(facecolor="#d62728", label="vraiment mort"),
], loc="upper center", bbox_to_anchor=(0.5, -0.03), fontsize=7, ncol=3, frameon=False)

erreur_img = axes[2].imshow(erreur.reshape(28, 28), cmap="inferno")
axes[2].set_title("Erreur quadratique par pixel\n(sortie ReLU)", fontsize=9)
fig.colorbar(erreur_img, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
_finish_figure(fig, "MNIST: les pixels tues par une sortie ReLU",
               describe_autoencoder(relu_run["model"], n_train=len(X_tr)), layout=False)

## Partie B - shapes

In [ ]:
X_shapes_train, y_shapes_train, shape_names = load_shapes_npz(split="train", max_samples=12000)
X_shapes_eval, y_shapes_eval, _ = load_shapes_npz(split="validation", max_samples=3000)
X_shapes_train = X_shapes_train.reshape(len(X_shapes_train), -1)
X_shapes_eval = X_shapes_eval.reshape(len(X_shapes_eval), -1)

SHAPES_SHAPE = (3, 32, 32)
SHAPES_DIM = 3 * 32 * 32
print("train:", X_shapes_train.shape, "| eval:", X_shapes_eval.shape, "| classes:", shape_names)
show_image_grid(X_shapes_eval[:8], SHAPES_SHAPE, nrow=1, ncol=8, title="Exemples de formes")

### Train, compression et decompression

In [ ]:
shapes_model = AutoEncoder(input_dim=SHAPES_DIM, output_dim=SHAPES_DIM, latent_dim=2,
                           encoder_layer_num=3, decoder_layer_num=3, encoder_activation_function=nn.ReLU)
shapes_model.fit(X_shapes_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

SHAPES_BASELINE_CONFIG = describe_autoencoder(shapes_model, n_train=len(X_shapes_train))

fig = plt.figure(figsize=(5, 3))
plt.plot(shapes_model.loss_history, marker="o")
plt.xlabel("epoch"); plt.ylabel("MSE d'entrainement")
_finish_figure(fig, "Courbe d'apprentissage shapes",
               describe_autoencoder(shapes_model, n_train=len(X_shapes_train), omit=("epochs",)))

shapes_latent = shapes_model.encode(X_shapes_eval)
shapes_reconstructed = shapes_model.decode(shapes_latent)
show_original_vs_reconstruction_grid(X_shapes_eval, shapes_reconstructed, SHAPES_SHAPE, n=8,
                     title="Shapes: original (haut) vs reconstruit (bas)",
                     config=SHAPES_BASELINE_CONFIG)

### Metriques de compression

In [ ]:
shapes_report = compression_report(shapes_model.get_codebook(), shapes_latent, X_shapes_eval, shapes_reconstructed)
print_compression_report(shapes_report)

per_image_bytes = shapes_latent.n_bytes / len(X_shapes_eval)
print(f"\nMessage par image: {per_image_bytes:.0f} octets contre {X_shapes_eval[0].nbytes} octets en float32.")

### Espace latent et generation

In [ ]:
plot_latent_scatter(shapes_latent.array, y_shapes_eval, class_names=shape_names,
                    title="Espace latent 2D des formes",
                    config=SHAPES_BASELINE_CONFIG)

# Generation par echantillonnage gaussien
new_codes = sample_gaussian_latent(shapes_latent.array, n_samples=16)
generated = shapes_model.decode(Latent(array=new_codes, nature="continuous"))
show_image_grid(generated, SHAPES_SHAPE, nrow=2, ncol=8,
                title="Formes generees (echantillonnage gaussien)",
                config=SHAPES_BASELINE_CONFIG)

# Interpolation entre deux formes
path = interpolate_latent(shapes_latent.array[0], shapes_latent.array[5], steps=10)
morph = shapes_model.decode(Latent(array=path, nature="continuous"))
show_image_grid(morph, SHAPES_SHAPE, nrow=1, ncol=10,
                title="Interpolation entre deux formes",
                config=SHAPES_BASELINE_CONFIG)

### Experimentation hyper-parametres

In [ ]:
X_shapes_tr, _ = subsample_dataset(X_shapes_train, y_shapes_train, 6000, seed=1)
shapes_runs = {}
shapes_results = []
for latent_dim in latent_dims:
    run = run_autoencoder_hyperparam_experiment(X_shapes_tr, X_shapes_eval, SHAPES_DIM, latent_dim, nn.ReLU, EPOCHS_SWEEP)
    shapes_runs[latent_dim] = run
    report = run["report"]
    report.update(latent_dim=latent_dim)
    shapes_results.append(report)
    print(f"latent={latent_dim:>2} | MSE={report['reconstruction_mse']:.4f} "
          f"| total={report['total_compressed_bytes']:>9,} o | ratio={report['compression_ratio']:.2f}")

# Explore ici: la dimension latente (axe x, puis etiquettes de lignes).
shapes_sweep_config = describe_autoencoder(
    shapes_runs[latent_dims[0]]["model"], n_train=len(X_shapes_tr),
    omit=("couches",), extra=f"couches=geomspace({SHAPES_DIM}->latent, 3+3)",
)

fig = plt.figure(figsize=(5, 4))
plt.plot([r["latent_dim"] for r in shapes_results], [r["reconstruction_mse"] for r in shapes_results], marker="o")
plt.xlabel("dimension latente"); plt.ylabel("MSE")
_finish_figure(fig, "Shapes: qualite vs dimension latente", shapes_sweep_config)

# Decompression et generation selon la dimension latente
show_labeled_image_rows(
    [X_shapes_eval] + [shapes_runs[d]["reconstruction"] for d in latent_dims],
    SHAPES_SHAPE,
    ["original"] + [f"latent={d}\n{shapes_runs[d]['report']['reconstruction_mse']:.4f}" for d in latent_dims],
    n=8, title="Shapes: decompression selon la dimension latente (etiquette = MSE)",
    config=shapes_sweep_config,
)

show_labeled_image_rows(
    [generate_from_latent_using_gaussian(shapes_runs[d]["model"], shapes_runs[d]["latent"], 8, seed=1) for d in latent_dims],
    SHAPES_SHAPE,
    [f"latent={d}" for d in latent_dims],
    n=8, title="Shapes: generation par echantillonnage gaussien selon la dimension latente",
    config=shapes_sweep_config,
)

### Variation de la fonction de perte
MSE, L1, BCE

In [ ]:
shapes_loss_mse = {}
shapes_loss_config = None
for name, loss_cls in loss_functions.items():
    model = AutoEncoder(input_dim=SHAPES_DIM, output_dim=SHAPES_DIM, latent_dim=2,
                        encoder_layer_num=3, decoder_layer_num=3,
                        encoder_activation_function=nn.ReLU, loss_function=loss_cls)
    model.fit(X_shapes_tr, epochs=EPOCHS_SWEEP, batch_size=BATCH_SIZE)
    latent = model.encode(X_shapes_eval)
    reconstruction = model.decode(latent)
    shapes_loss_mse[name] = compression_report(model.get_codebook(), latent, X_shapes_eval, reconstruction)["reconstruction_mse"]
    shapes_loss_config = describe_autoencoder(model, n_train=len(X_shapes_tr), omit=("loss",))
    show_original_vs_reconstruction_grid(X_shapes_eval, reconstruction, SHAPES_SHAPE, n=8,
                         title=f"Shapes reconstruit - perte {name} (MSE d'evaluation={shapes_loss_mse[name]:.4f})",
                         config=shapes_loss_config)

fig = plt.figure(figsize=(4.5, 3))
plt.bar(list(shapes_loss_mse), list(shapes_loss_mse.values()))
plt.ylabel("MSE d'evaluation")
_finish_figure(fig, "Shapes: impact de la fonction de perte", shapes_loss_config)

In [ ]:
shapes_loss_mse = {}
shapes_loss_config = None
for name, loss_cls in loss_functions.items():
    model = AutoEncoder(input_dim=SHAPES_DIM, output_dim=SHAPES_DIM, latent_dim=32,
                        encoder_layer_num=3, decoder_layer_num=3,
                        encoder_activation_function=nn.ReLU, loss_function=loss_cls)
    model.fit(X_shapes_tr, epochs=EPOCHS_SWEEP, batch_size=BATCH_SIZE)
    latent = model.encode(X_shapes_eval)
    reconstruction = model.decode(latent)
    shapes_loss_mse[name] = compression_report(model.get_codebook(), latent, X_shapes_eval, reconstruction)["reconstruction_mse"]
    shapes_loss_config = describe_autoencoder(model, n_train=len(X_shapes_tr), omit=("loss",))
    show_original_vs_reconstruction_grid(X_shapes_eval, reconstruction, SHAPES_SHAPE, n=8,
                         title=f"Shapes reconstruit - perte {name} (MSE d'evaluation={shapes_loss_mse[name]:.4f})",
                         config=shapes_loss_config)

fig = plt.figure(figsize=(4.5, 3))
plt.bar(list(shapes_loss_mse), list(shapes_loss_mse.values()))
plt.ylabel("MSE d'evaluation")
_finish_figure(fig, "Shapes: impact de la fonction de perte", shapes_loss_config)

## Experimentation - nombre d'epochs

Chaque budget entraine un modele independant plutot que de prolonger le precedent: `fit`
reinstancie son optimiseur Adam a chaque appel, donc 4 appels de 20 epochs ne seraient pas
equivalents a un seul de 80.

In [ ]:
EPOCH_BUDGETS = [5, 10, 20, 40, 80]
EPOCH_EXP_LATENT = 2

epoch_runs = {}
for budget in EPOCH_BUDGETS:
    run = run_autoencoder_hyperparam_experiment(
        X_tr, X_mnist_eval, 784, EPOCH_EXP_LATENT, nn.LeakyReLU, budget, loss_function=nn.BCELoss
    )
    # MSE d'entrainement: comparable a la MSE d'evaluation, contrairement a la perte BCE.
    train_reconstruction = run["model"].decode(run["model"].encode(X_tr))
    run["train_mse"] = float(np.mean((X_tr - train_reconstruction) ** 2))
    epoch_runs[budget] = run
    print(f"epochs={budget:>3} | MSE train={run['train_mse']:.4f} | MSE eval={run['report']['reconstruction_mse']:.4f} "
          f"| ecart={run['report']['reconstruction_mse'] - run['train_mse']:+.4f} "
          f"| BCE finale={run['model'].loss_history[-1]:.4f}")

best_budget = min(EPOCH_BUDGETS, key=lambda b: epoch_runs[b]["report"]["reconstruction_mse"])
print(f"\nMeilleur budget sur la MSE d'evaluation: {best_budget} epochs")

# Explore ici: le nombre d'epochs (axe x / etiquettes de lignes) -> retire de la note.
epochs_config = describe_autoencoder(
    epoch_runs[EPOCH_BUDGETS[-1]]["model"], n_train=len(X_tr), omit=("epochs",)
)

# 1. Sous- ou sur-apprentissage: les deux MSE dans la meme unite.
fig = plt.figure(figsize=(5.5, 4))
plt.plot(EPOCH_BUDGETS, [epoch_runs[b]["train_mse"] for b in EPOCH_BUDGETS],
         marker="o", label="MSE d'entrainement")
plt.plot(EPOCH_BUDGETS, [epoch_runs[b]["report"]["reconstruction_mse"] for b in EPOCH_BUDGETS],
         marker="o", label="MSE d'evaluation")
plt.axvline(best_budget, color="0.6", linestyle="--", linewidth=1)
plt.xscale("log"); plt.xticks(EPOCH_BUDGETS, [str(b) for b in EPOCH_BUDGETS])
plt.xlabel("nombre d'epochs"); plt.ylabel("MSE"); plt.legend()
_finish_figure(fig, "MNIST: qualite vs budget d'entrainement", epochs_config)

# 2. Vue optimisation: la perte reellement minimisee, sur le run le plus long.
longest_model = epoch_runs[EPOCH_BUDGETS[-1]]["model"]
fig = plt.figure(figsize=(5.5, 3))
plt.plot(range(1, len(longest_model.loss_history) + 1), longest_model.loss_history, marker=".")
plt.xlabel("epoch"); plt.ylabel("perte BCE d'entrainement")
_finish_figure(fig, f"Courbe d'apprentissage BCE (run le plus long: {EPOCH_BUDGETS[-1]} epochs)",
               describe_autoencoder(longest_model, n_train=len(X_tr), omit=("epochs",)))

# 3. Effet visuel du budget sur la decompression puis la generation.
show_labeled_image_rows(
    [X_mnist_eval] + [epoch_runs[b]["reconstruction"] for b in EPOCH_BUDGETS],
    MNIST_SHAPE,
    ["original"] + [f"epochs={b}\n{epoch_runs[b]['report']['reconstruction_mse']:.4f}" for b in EPOCH_BUDGETS],
    n=8, title="MNIST: decompression selon le budget d'entrainement (etiquette = MSE eval)",
    config=epochs_config,
)

show_labeled_image_rows(
    [generate_from_latent_using_gaussian(epoch_runs[b]["model"], epoch_runs[b]["latent"], 8, seed=1)
     for b in EPOCH_BUDGETS],
    MNIST_SHAPE,
    [f"epochs={b}" for b in EPOCH_BUDGETS],
    n=8, title="MNIST: generation par echantillonnage gaussien selon le budget d'entrainement",
    config=epochs_config,
)

In [ ]:
EPOCH_BUDGETS = [5, 10, 20, 40, 80]
EPOCH_EXP_LATENT = 4

epoch_runs = {}
for budget in EPOCH_BUDGETS:
    run = run_autoencoder_hyperparam_experiment(
        X_tr, X_mnist_eval, 784, EPOCH_EXP_LATENT, nn.LeakyReLU, budget, loss_function=nn.BCELoss
    )
    # MSE d'entrainement: comparable a la MSE d'evaluation, contrairement a la perte BCE.
    train_reconstruction = run["model"].decode(run["model"].encode(X_tr))
    run["train_mse"] = float(np.mean((X_tr - train_reconstruction) ** 2))
    epoch_runs[budget] = run
    print(f"epochs={budget:>3} | MSE train={run['train_mse']:.4f} | MSE eval={run['report']['reconstruction_mse']:.4f} "
          f"| ecart={run['report']['reconstruction_mse'] - run['train_mse']:+.4f} "
          f"| BCE finale={run['model'].loss_history[-1]:.4f}")

best_budget = min(EPOCH_BUDGETS, key=lambda b: epoch_runs[b]["report"]["reconstruction_mse"])
print(f"\nMeilleur budget sur la MSE d'evaluation: {best_budget} epochs")

# Explore ici: le nombre d'epochs (axe x / etiquettes de lignes) -> retire de la note.
epochs_config = describe_autoencoder(
    epoch_runs[EPOCH_BUDGETS[-1]]["model"], n_train=len(X_tr), omit=("epochs",)
)

# 1. Sous- ou sur-apprentissage: les deux MSE dans la meme unite.
fig = plt.figure(figsize=(5.5, 4))
plt.plot(EPOCH_BUDGETS, [epoch_runs[b]["train_mse"] for b in EPOCH_BUDGETS],
         marker="o", label="MSE d'entrainement")
plt.plot(EPOCH_BUDGETS, [epoch_runs[b]["report"]["reconstruction_mse"] for b in EPOCH_BUDGETS],
         marker="o", label="MSE d'evaluation")
plt.axvline(best_budget, color="0.6", linestyle="--", linewidth=1)
plt.xscale("log"); plt.xticks(EPOCH_BUDGETS, [str(b) for b in EPOCH_BUDGETS])
plt.xlabel("nombre d'epochs"); plt.ylabel("MSE"); plt.legend()
_finish_figure(fig, "MNIST: qualite vs budget d'entrainement", epochs_config)

# 2. Vue optimisation: la perte reellement minimisee, sur le run le plus long.
longest_model = epoch_runs[EPOCH_BUDGETS[-1]]["model"]
fig = plt.figure(figsize=(5.5, 3))
plt.plot(range(1, len(longest_model.loss_history) + 1), longest_model.loss_history, marker=".")
plt.xlabel("epoch"); plt.ylabel("perte BCE d'entrainement")
_finish_figure(fig, f"Courbe d'apprentissage BCE (run le plus long: {EPOCH_BUDGETS[-1]} epochs)",
               describe_autoencoder(longest_model, n_train=len(X_tr), omit=("epochs",)))

# 3. Effet visuel du budget sur la decompression puis la generation.
show_labeled_image_rows(
    [X_mnist_eval] + [epoch_runs[b]["reconstruction"] for b in EPOCH_BUDGETS],
    MNIST_SHAPE,
    ["original"] + [f"epochs={b}\n{epoch_runs[b]['report']['reconstruction_mse']:.4f}" for b in EPOCH_BUDGETS],
    n=8, title="MNIST: decompression selon le budget d'entrainement (etiquette = MSE eval)",
    config=epochs_config,
)

show_labeled_image_rows(
    [generate_from_latent_using_gaussian(epoch_runs[b]["model"], epoch_runs[b]["latent"], 8, seed=1)
     for b in EPOCH_BUDGETS],
    MNIST_SHAPE,
    [f"epochs={b}" for b in EPOCH_BUDGETS],
    n=8, title="MNIST: generation par echantillonnage gaussien selon le budget d'entrainement",
    config=epochs_config,
)

## Experimentation - activation du code latent

Jusqu'ici le code latent est toujours la sortie brute d'un `nn.Linear`: `build_layers` ne pose
d'activation qu'*entre* les couches, jamais sur la derniere. L'espace latent est donc non borne,
et c'est ce qui autorise les codes a s'etaler a n'importe quelle echelle.

`AutoEncoder` accepte maintenant `latent_activation`, appliquee a la sortie de l'encodeur: le
code devient `act(Linear(...))`. On compare ici trois latents a dimension 2 (la seule
directement visualisable), tout le reste etant fixe.

Ce qu'il faut regarder:

- **lineaire** (defaut): nuage non borne, notre reference.
- **ReLU**: le code ne peut plus etre negatif, le nuage est confine au quadrant positif. Les
  unites mortes sortent exactement 0, donc des codes s'ecrasent sur les axes: la moitie du plan
  est inutilisable.
- **Sigmoid**: le code est ecrase dans (0,1) et sature vers les bords. C'est aussi la variante
  qui etrangle le plus le gradient, donc on attend la pire MSE.

Le point le plus interessant est la generation. `sample_gaussian_latent` ajuste une gaussienne
*non bornee* (`np.cov` + `multivariate_normal`). Face a un nuage borne (Sigmoid) ou unilateral
(ReLU), une bonne part des codes tires tombe hors de ce que l'encodeur peut produire: le
decodeur n'y a jamais ete entraine, et les images se degradent. Ce n'est pas un bug a corriger
en tronquant dans `sample_gaussian_latent` - ce serait modifier en silence les figures
precedentes - c'est le resultat de l'experience.

In [ ]:
latent_activations = {"lineaire": None, "ReLU": nn.ReLU, "Sigmoid": nn.Sigmoid}
LATENT_ACT_DIM = 2

latent_act_runs = {}
for act_name, act in latent_activations.items():
    run = run_autoencoder_hyperparam_experiment(
        X_tr, X_mnist_eval, 784, LATENT_ACT_DIM, nn.ReLU, EPOCHS_SWEEP,
        loss_function=nn.MSELoss, latent_activation_function=act,
    )
    codes = run["latent"].array
    latent_act_runs[act_name] = run
    print(f"{act_name:>8} | MSE={run['report']['reconstruction_mse']:.4f} "
          f"| z1 dans [{codes[:, 0].min():+.2f}, {codes[:, 0].max():+.2f}] "
          f"| z2 dans [{codes[:, 1].min():+.2f}, {codes[:, 1].max():+.2f}] "
          f"| codes exactement nuls: {100 * np.mean(codes == 0):.1f}%")

# Part des codes gaussiens qui tombent hors du domaine atteignable par l'encodeur.
for act_name, act in latent_activations.items():
    if act is None:
        continue
    codes = latent_act_runs[act_name]["latent"].array
    sampled = sample_gaussian_latent(codes, 2000, seed=1)
    low, high = (0.0, 1.0) if act is nn.Sigmoid else (0.0, np.inf)
    outside = np.mean(np.any((sampled < low) | (sampled > high), axis=1))
    print(f"{act_name:>8} | codes gaussiens hors domaine atteignable: {100 * outside:.1f}%")

# Explore ici: l'activation latente (titre de chaque panneau) -> retiree de la note.
latent_act_config = describe_autoencoder(
    latent_act_runs["lineaire"]["model"], n_train=len(X_tr), omit=("latent_act",)
)

fig, axes = plt.subplots(1, len(latent_activations),
                         figsize=(3.6 * len(latent_activations), 3.9), layout="constrained")
for ax, act_name in zip(axes, latent_activations):
    run = latent_act_runs[act_name]
    codes = run["latent"].array
    scatter = ax.scatter(codes[:, 0], codes[:, 1], c=y_mnist_eval, cmap="tab10", s=4, alpha=0.6)
    ax.set_title(f"latent = {act_name} (MSE={run['report']['reconstruction_mse']:.4f})", fontsize=10)
    ax.set_xlabel("z1"); ax.set_ylabel("z2")
fig.colorbar(scatter, ax=list(axes), label="chiffre")
_finish_figure(fig, "MNIST: geometrie du code latent selon son activation",
               latent_act_config, layout=False)

show_labeled_image_rows(
    [X_mnist_eval] + [latent_act_runs[a]["reconstruction"] for a in latent_activations],
    MNIST_SHAPE,
    ["original"] + [f"{a}\n{latent_act_runs[a]['report']['reconstruction_mse']:.4f}" for a in latent_activations],
    n=8, title="MNIST: decompression selon l'activation du code latent (etiquette = MSE)",
    config=latent_act_config,
)

show_labeled_image_rows(
    [generate_from_latent_using_gaussian(latent_act_runs[a]["model"], latent_act_runs[a]["latent"], 8, seed=1)
     for a in latent_activations],
    MNIST_SHAPE,
    list(latent_activations),
    n=8, title="MNIST: generation par echantillonnage gaussien selon l'activation du code latent",
    config=latent_act_config,
)

## Experimentation - profondeur du reseau a latent fixe

Toutes les experiences precedentes utilisent 3 couches d'encodeur et 3 de decodeur. On fixe
ici la dimension latente a 2 et on fait varier la profondeur de 1 a 6, pour voir comment la
qualite de compression depend du nombre de couches.

Deux modes, car ils ne coutent pas la meme chose:

- **symetrique** (`encoder_layer_num = decoder_layer_num = d`): l'histoire debit-distorsion
  honnete, notre resultat principal.
- **encodeur seul** (`decoder_layer_num` reste a 3): le temoin. Comme `get_codebook()` ne
  renvoie que les poids du **decodeur**, approfondir l'encodeur **ne coute rien** en
  transmission - l'encodeur n'est jamais envoye. Ce temoin isole donc l'effet gratuit.

Trois points de lecture:

1. **Le codebook explose, le message non.** A latent 2, chaque image coute toujours 8 octets,
   quelle que soit la profondeur. Mais le decodeur passe de 9 408 octets (d=1) a 1 054 144
   octets (d=6), soit x112. Le ratio de compression s'effondre de ~280 a ~9.
2. **"Plus profond = meilleur" est faux.** La MSE atteint son minimum vers d=3 puis **remonte**.
   Un reseau profond et etroit (une couche a 5 unites) avec un goulot a 2 dimensions est
   difficile a optimiser.
3. **La variance entre graines explose** (environ x20 entre d=1 et d=5). Une courbe a une seule
   graine invente des accidents: on y voit un faux creux qui disparait en moyennant. D'ou les
   3 graines et les barres d'erreur - **l'instabilite est elle-meme un resultat**: le cout de la
   profondeur n'est pas seulement en octets, il est en entrainabilite.

A noter: a `d=1`, `build_layers` ne pose **aucune activation cachee** (la condition
`i < len(sizes)-2` est fausse pour `[784, 2]`), donc l'encodeur est **purement lineaire** -
la note de la figure l'annonce d'ailleurs avec `act=aucune`. C'est essentiellement une PCA,
la comparaison chiffree est faite dans `02_pca.ipynb`. Attention toutefois: le decodeur reste
`Sigmoid(Linear(z))`, ce n'est donc pas exactement une PCA.

In [ ]:
DEPTHS = [1, 2, 3, 4, 5, 6]
DEPTH_SEEDS = [0, 1, 2]
DEPTH_LATENT = 2
# "encodeur seul" garde le decodeur a 3 couches: le codebook (= poids du decodeur) reste
# constant, ce qui isole l'effet de la profondeur d'encodeur, gratuite en transmission.
DEPTH_MODES = {"symetrique": None, "encodeur seul": 3}

depth_results = {}
for mode_name, decoder_fixe in DEPTH_MODES.items():
    for depth in DEPTHS:
        mses, report = [], None
        for seed in DEPTH_SEEDS:
            torch.manual_seed(seed)  # le helper n'expose pas de graine: on la pose ici
            run = run_autoencoder_hyperparam_experiment(
                X_tr, X_mnist_eval, 784, DEPTH_LATENT, nn.ReLU, EPOCHS_SWEEP,
                encoder_layer_num=depth,
                decoder_layer_num=depth if decoder_fixe is None else decoder_fixe,
            )
            mses.append(run["report"]["reconstruction_mse"])
            report = run["report"]
        depth_results[(mode_name, depth)] = {
            "mse_moyenne": float(np.mean(mses)), "mse_ecart": float(np.std(mses)),
            "codebook": report["codebook_bytes"], "total": report["total_compressed_bytes"],
            "ratio": report["compression_ratio"],
        }
        r = depth_results[(mode_name, depth)]
        print(f"{mode_name:>13} | d={depth} | MSE={r['mse_moyenne']:.4f} +/- {r['mse_ecart']:.4f} "
              f"| codebook={r['codebook']:>9,} o | ratio={r['ratio']:6.2f}")
    print()

meilleure = min(DEPTHS, key=lambda d: depth_results[("symetrique", d)]["mse_moyenne"])
print(f"Meilleure profondeur (symetrique, MSE moyenne): d={meilleure}")

# Explore ici: la profondeur (axe x) et le mode (legende). `couches` decrit l'encodeur seul,
# donc il induirait en erreur en mode "encodeur seul": on donne les deux regles a la place.
depth_config = describe_autoencoder(
    mnist_runs[("ReLU", 2)]["model"], n_train=len(X_tr),
    omit=("couches", "epochs"),
    extra=f"encodeur=geomspace(784->2, d) | latent={DEPTH_LATENT} | epochs={EPOCHS_SWEEP} | {len(DEPTH_SEEDS)} graines",
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for mode_name in DEPTH_MODES:
    moyennes = [depth_results[(mode_name, d)]["mse_moyenne"] for d in DEPTHS]
    ecarts = [depth_results[(mode_name, d)]["mse_ecart"] for d in DEPTHS]
    totaux = [depth_results[(mode_name, d)]["total"] for d in DEPTHS]
    axes[0].errorbar(DEPTHS, moyennes, yerr=ecarts, marker="o", capsize=3, label=mode_name)
    axes[1].errorbar(totaux, moyennes, yerr=ecarts, marker="o", capsize=3, label=mode_name)
axes[0].axvline(meilleure, color="0.6", linestyle="--", linewidth=1)
axes[0].set_xlabel("profondeur (nb de couches)"); axes[0].set_ylabel("MSE d'evaluation")
axes[0].set_title("Qualite vs profondeur (moyenne +/- ecart-type)"); axes[0].legend()
axes[1].set_xscale("log")
axes[1].set_xlabel("taille compressee totale (octets, echelle log)"); axes[1].set_ylabel("MSE d'evaluation")
axes[1].set_title("Compromis taille / qualite"); axes[1].legend()
_finish_figure(fig, "MNIST: profondeur a dimension latente fixee (2)", depth_config)

## Reponses aux questions du projet

1. Nature de l'espace latent.

Il est continu: encode renvoie un Latent de nature "continuous", c'est-a-dire un vecteur de reels (float32) dense. N'importe quel point de l'espace peut etre decode, ce qui rend possibles l'interpolation et l'echantillonnage. Cela contraste avec K-Means, dont l'espace latent est discret (un indice de cluster parmi K).

2. Codebook.

Le codebook renvoye par get_codebook regroupe les poids du decodeur. C'est le dictionnaire partage: l'emetteur et le recepteur doivent tous deux en disposer pour reconstruire. Concretement, l'emetteur compresse avec l'encodeur et n'envoie que le code latent; le recepteur applique le decodeur (le codebook) a ce code. Le codebook est un cout fixe, paye une seule fois et amorti sur toutes les images.

3. Qualite de reconstruction.

Mesuree par la MSE de compression_report (voir les tableaux ci-dessus) et visible sur les grilles original vs reconstruit. Elle s'ameliore quand la dimension latente augmente, au prix d'un message plus gros: c'est le compromis illustre par les courbes d'experimentation. Elle depend aussi de la fonction de perte d'entrainement (MSE, L1, BCE).

4. Code byte (taille du message).

Le message transmis par image est le code latent: dimension_latente x 4 octets (float32), soit par exemple 128 octets pour une dimension latente de 32. La taille totale pour un lot est latent_bytes dans compression_report; le codebook s'y ajoute une seule fois, independamment du nombre d'images.